In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

In [5]:
db = SQLDatabase.from_uri("sqlite:///chinook.db")
def get_schema(_):
    return db.get_table_info()

def run_query(query):
    print(f'Query being run : {query} \n\n')
    return db.run(query)


In [8]:
print(get_schema(None))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

In [49]:

from langchain_core.messages import HumanMessage
from langchain_core.prompts import MessagesPlaceholder

# Open the file first so 'f' is defined before parsing lines
chat_history = []
with open('chat_history.txt', 'r') as f:
    chat_history = [HumanMessage(content=line.strip()) for line in f if line.strip()]

print(chat_history)

# Replace the community import with the dedicated package
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_openai import ChatOpenAI

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

def get_llm():
    endpoint = HuggingFaceEndpoint(
        repo_id="Qwen/Qwen2.5-72B-Instruct",
        task="text-generation",
        temperature=0.1,
        max_new_tokens=512,
    )
    return ChatHuggingFace(llm=endpoint)
    
def write_sql_query(llm):
    template = """Based on the table schema below, write a SQL query that would answer the user's question:
    {schema}

    Question: {question}
    SQL Query:"""

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Given an input question, convert it to a SQL query. No pre-amble. "
                "Please do not return anything else apart from the SQL query, no prefix or suffix quotes, no sql keyword, nothing please."
            ),
            MessagesPlaceholder(variable_name='chat_history'),
            ("human", template),
        ]
    )

    return (
        RunnablePassthrough.assign(schema=get_schema)
        | prompt
        | llm
        | StrOutputParser()
    )

[]


In [25]:
def answer_user_query(query, llm, history):
    template = """Based on the table schema below, question, sql query, and sql response, write a natural language response:
    {schema}

    Question: {question}
    SQL Query: {query}
    SQL Response: {response}"""

    prompt_response = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Given an input question and SQL response, convert it to a natural language answer. No pre-amble.",
            ),
            ("human", template),
        ]
    )

    # We use a lambda to explicitly forward 'x' (which contains question AND chat_history) to write_sql_query
    sql_chain = write_sql_query(llm)

    full_chain = (
        RunnablePassthrough.assign(schema=get_schema)
        | RunnablePassthrough.assign(query=lambda x: sql_chain.invoke(x))
        | RunnablePassthrough.assign(response=lambda x: run_query(x["query"]))
        | prompt_response
        | llm
        | StrOutputParser()
    )

    return full_chain.invoke({
        "question": query,
        "chat_history": history
    })

In [50]:
load_dotenv()
query = "give me name of 10 artists"
    
    # 1. Pass the chat history into answer_user_query
    # 2. StrOutputParser returns a string directly, so print(response) is used instead of response.content
response = answer_user_query(query, llm=get_llm(), history=chat_history)
print(response)

Query being run : SELECT Name FROM Artist LIMIT 10 


The names of 10 artists are: AC/DC, Accept, Aerosmith, Alanis Morissette, Alice In Chains, Antônio Carlos Jobim, Apocalyptica, Audioslave, BackBeat, and Billy Cobham.
